## ZADANIE 1 ##
Cezary Kalarus

## DANE: ##
https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
<br>Ale ucięte do tylko rap i rock 

## Hipoteza ##
<br>Model głównie będzie rozróżniać gatunek między rapem (hip-hop w datasecie) a rockiem patrząc na speechiness (ratio mowy w nagraniu) i explicit (boolean, czy występują przekleństwa i inne nieodpowiednie dla dzieci rzeczy). Jeśli zabierzemy te dane modelowi, to jego skuteczność spadnie, ale powinien być w stanie nadal w miarę dobrze klasyfikować po rytmie i brzmieniu (danceability, energy, valence)

## Początkowe obserwacje ##
Najpierw popatrzmy na nasz zbiór aby wyciągnąć jakieś wnioski z niego (nie będziemy uwzględniać sub-genre rocka, jest ich za dużo, a hip-hop tylko jeden, da to okropne ratio rap:rock i za duże różnice w cechach tych podgatunkow)

In [15]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

df = pd.read_csv("dataset.csv")

# Ucinamy dataset z innych genre
df = df[df["track_genre"].isin(["rock", "hip-hop"])]
# Usuwamy duplikaty - w spotify np często ta sama piosenka ma różne wydania, przez co będzie się dublowac
df = df.drop_duplicates(subset=["artists", "track_name"])
# W datasecie sa tez kawalki zespolow jak 

print("<<< Liczba utworow po oczyszczeniu >>>")
print(df["track_genre"].value_counts())

print("<<< Avg, mediana i maksymalne odchylenia w speechiness (rock a hiphop) >>>")
print(df.groupby("track_genre")["speechiness"].agg(["mean", "median", "min", "max"]).round(3))

print("<<< Avg, mediana i maksymalne odchylenia w explicit (rock a hiphop) (to boolean!!)>>>")
print(df.groupby("track_genre")["explicit"].agg(["mean", "median", "min", "max"]).round(3))


cechy = ["speechiness", "danceability", "energy", "acousticness", "valence"]
print("<<< Avg dla pozostalych cech >>>")
print(df.groupby("track_genre")[cechy].mean().round(3))

<<< Liczba utworow po oczyszczeniu >>>
track_genre
hip-hop    733
rock       572
Name: count, dtype: int64
<<< Avg, mediana i maksymalne odchylenia w speechiness (rock a hiphop) >>>
              mean  median    min    max
track_genre                             
hip-hop      0.137   0.096  0.024  0.621
rock         0.053   0.039  0.024  0.391
<<< Avg, mediana i maksymalne odchylenia w explicit (rock a hiphop) (to boolean!!)>>>
              mean  median    min   max
track_genre                            
hip-hop      0.247     0.0  False  True
rock         0.056     0.0  False  True
<<< Avg dla pozostalych cech >>>
             speechiness  danceability  energy  acousticness  valence
track_genre                                                          
hip-hop            0.137         0.718   0.690         0.220    0.544
rock               0.053         0.556   0.683         0.208    0.526


### Pierwsze wnioski ###
- speechiness (0 - 1) - proporcja mowy w nagraniu (w rapie będzie o dużo wyższa) wg. Spotify:
    - <= 0.33 - zwykła muzyka, melodyjna, brak samej mowy 
    - (0.33; 0.66) - muzyka z obecną mową (jak rap właśnie)
    -  \>= 0.66 - podcasty (głównie mowa)
<br>
![](spotify-docs-ss.png)
(https://developer.spotify.com/documentation/web-api/reference/get-audio-features)
<br>

#### W naszym datasecie speechiness wypada kompletnie inaczej: ####
obie średnie są poniżej .33 (wg spotify rap powinien być nad). Dla obu gatunków minima są takie same - pewnie jakieś interlude/ballada. Maksimum dla rocka podchodzi już (wg spotify) pod rap, a maksimum rapu już prawie pod podcast. 

- explicit: rap ma niemal 5-krotnie większą średnią (skoro to boolean to avg poprostu zlicza ile jest kawałków oznakowanych explicit) (więc też skoro to boolean, to to kryterium samo w sobie, bez udziału innych nie jest dobre). Mediana, maksima i minima nic nam nie mówią - w każdej kategorii znajdzą się utwory, które będą i nie będą explicit - kod znalazł po prostu któreś z nich.

Sprawdźmy zatem jakie utwory i jakich wykonawców najbardziej odbiegają od norm speechiness

In [16]:
kolumny_info = ["artists", "track_name", "speechiness", "danceability", "explicit"]

rap = df[df["track_genre"] == "hip-hop"]
rock = df[df["track_genre"] == "rock"]

print("<<< HIP-HOP: najwyzsze speechiness >>>")
print(rap.nlargest(10, "speechiness")[kolumny_info])

print("<<< HIP-HOP: najnizsze speechiness >>>")
print(rap.nsmallest(10, "speechiness")[kolumny_info])

print("<<< ROCK: najwyzsze speechiness >>>")
print(rock.nlargest(10, "speechiness")[kolumny_info])

print("<<< ROCK: najnizsze speechiness >>>")
print(rock.nsmallest(10, "speechiness")[kolumny_info])

<<< HIP-HOP: najwyzsze speechiness >>>
                                       artists              track_name  speechiness  danceability  explicit
51805         DIVINE;Aavrutti;D'Evil;Shah Rule                Vibe Hai        0.621         0.646     False
51807  Young Stunners;Talha Anjum;Talhah Yunus                 Afsanay        0.569         0.816     False
51969                                 Don Omar                   Bomba        0.485         0.661     False
51536                                   DaBaby       Ball If I Want To        0.457         0.902      True
51083                                   DIVINE  Kaam 25 - Sacred Games        0.451         0.484     False
51200                                   Future                Mask Off        0.431         0.833      True
51694              Anirudh Ravichander;Dhanush         What a Karavaad        0.428         0.431     False
51446                 Jack Harlow;Pooh Shiesty   SUVs (Black on Black)        0.425         0.813

### Wnioski z tego ###
- Bardzo dużo hinduskich kawałków (MaharshiPandya - autor to pewnie hindus i czytajac z api łączył sie z hinduskimi serwerami przez co to mu pobrało dużo tego)
- Hip-hop: 
    - Najwyższe speechiness ma wiele z tych hindusów ale też są DaBaby, Future, Jack Harlow
    - Najniżej ten wskaźnik mają jakieś hinduskie ballady, które jakkolwiek nie pasuja do tego gatunku. 
    - Pokazuje to, że ten dataset jest w sumie fest zanieczyszczony - przez to że spotify wrzuca do tej kategorii jakies ballady o speechiness ok. 0.02, to żaden model nie osiągnie 100% skuteczności na samym speechiness. Tego nie ma jak niestety usunąć - nie ma nic, za pomocą czego moglibyśmy determinować czy dany rekord jest właśnie takim "śmieciem", a nie będę usuwać tego na ślepo tylko po to żeby potwierdzić hipotezę (najwyżej to później zrobie jak nie zapomne, tylko żeby zobaczyć jak to zmieni wyniki)
- Rock: 
    - Najwyżej jest The Police - nigdy bym nie powiedział wsm. Ten algorytm, który to liczy nie jest pewnie idealny, ale też Roxanne i Cant Stand Losing You w sumie nie mają jakichś solówek właściwie, gitara gra tylko same uderzenia. Algorytm musiał widzieć brak ciagłych "ścian" dźwięku.
    - Wysoko są też Imagine Dragons - ale to pewnie dlatego, że współautorem jest JID - raper. Tutaj też idealnie widać powtórki - Enemy powtarza się w aż 3 różnych wersjach - i to obok siebie.
    - Najniższy wskaźnik tutaj mają ballady i piosenki z bardzo długimi tymi "ścianami" samego dźwięku. Dla rocka to jednak jest normalne, a nie zanieczyszczenie.

Usuńmy teraz te duplikaty - porównując regexowo podobieństwa w tytule i autorstwie oraz identyczne (lub niemal) parametry. Też w datasecie są utwory wpisane jednocześnie jako rock i hip-hop. To też usuniemy

In [17]:
# Usuwamy utwory wpisane do 2 gatunkow naraz
df_raw = pd.read_csv("dataset.csv")
df_raw = df_raw[df_raw["track_genre"].isin(["rock", "hip-hop"])]
w_obu = df_raw.groupby(["artists", "track_name"])["track_genre"].nunique()
sprzeczne = w_obu[w_obu > 1].index
df = df[~df.set_index(["artists", "track_name"]).index.isin(sprzeczne)].copy()

# Regexem z autorow bierzemy tylko tego pierwszego (i nim porównujemy), a z tytulow usuwamy rzeczy po ( lub '-' - tu zazwyczaj są napisane jakieś inne wersje
przed = len(df)
df["clean_artist"] = df["artists"].str.split(";").str[0].str.lower().str.strip()
df["clean_title"] = df["track_name"].str.replace(r"[\(\-].*", "", regex=True).str.lower().str.strip()
df = df.drop_duplicates(subset=["clean_artist", "clean_title"])

# Usuwamy duplikaty o identycznych parametrach 
cechy_audio = ["speechiness", "danceability", "energy", "valence", "acousticness"]
df = df.drop_duplicates(subset=cechy_audio)

print(f"Usunieto {przed - len(df)} ukrytych duplikatow")
print("<<< Liczba utworow po tym >>>")
print(df["track_genre"].value_counts())

print("<<< ROCK: najwyzsze speechiness teraz >>>")
print(df[df["track_genre"] == "rock"].nlargest(10, "speechiness")[kolumny_info])

print("<<< HIP-HOP: najnizsze speechiness teraz >>>")
print(df[df["track_genre"] == "hip-hop"].nsmallest(10, "speechiness")[kolumny_info])

Usunieto 38 ukrytych duplikatow
<<< Liczba utworow po tym >>>
track_genre
hip-hop    710
rock       554
Name: count, dtype: int64
<<< ROCK: najwyzsze speechiness teraz >>>
                                            artists                                         track_name  speechiness  danceability  explicit
91298                                    The Police                          Roxanne - Remastered 2003        0.391         0.615     False
91348                                    The Police           Can't Stand Losing You - Remastered 2003        0.290         0.736     False
91022  Imagine Dragons;JID;Arcane;League of Legends  Enemy (with JID) - from the series Arcane Leag...        0.266         0.728     False
91190                            The Rolling Stones                             Sympathy For The Devil        0.257         0.636     False
91483                                Alejandro Sanz                                    Te Canto Un Son        0.250         0.81

Teraz to już wygląda chyba dobrze. Popatrzmy jeszcze tylko czy nie ma nigdzie pustych komórek, i zobaczmy czy pozostałe dane, na które jeszcze nie patrzyliśmy, mają znaczenie. (Pomijamy kolumny techniczne i niezwiązane - id (pierwsze dwie kolumny o pustych nazwach), track_id (hash), album_name, popularity (nie zależy od gatunku), i kategoryczne - key, mode, time_signature (nie zależą jakkolwiek od gatunku praktycznie))

In [18]:
print("<<< Liczba pustych komorek w zbiorze >>>")
print(df.isna().sum().sum())

# Srednie dla pozostalych cech
pozostale_cechy = ["loudness", "tempo", "instrumentalness", "liveness", "duration_ms"]
print("<<< Avg dla pozostalych cech >>>")
print(df.groupby("track_genre")[pozostale_cechy].mean().round(3))

# Korelacja wszystkich cech z gatunkiem - 1 = hip-hop, 0 = rock
df["czy_hiphop"] = (df["track_genre"] == "hip-hop").astype(int)
df["explicit"] = df["explicit"].astype(int)

wszystkie_cechy = [
    "speechiness", "danceability", "explicit", "energy", "valence",
    "acousticness", "loudness", "tempo", "instrumentalness", "liveness", "duration_ms"
]

print("<<< Korelacja cech z gatunkiem (dodatnia -> w strone hip-hopu, ujemna - w strone rocka) >>>")
korelacje = df[wszystkie_cechy + ["czy_hiphop"]].corr()["czy_hiphop"].drop("czy_hiphop").sort_values(ascending=False)
print(korelacje.round(3))

<<< Liczba pustych komorek w zbiorze >>>
0
<<< Avg dla pozostalych cech >>>
             loudness    tempo  instrumentalness  liveness  duration_ms
track_genre                                                            
hip-hop        -6.126  118.581             0.013     0.199   210983.939
rock           -6.798  123.243             0.025     0.175   229226.166
<<< Korelacja cech z gatunkiem (dodatnia -> w strone hip-hopu, ujemna - w strone rocka) >>>
danceability        0.531
speechiness         0.456
explicit            0.257
loudness            0.134
liveness            0.079
valence             0.031
acousticness        0.026
energy              0.015
instrumentalness   -0.070
tempo              -0.077
duration_ms        -0.147
Name: czy_hiphop, dtype: float64


Nie ma żadnych pustych komórek. 
Za to, po korelacji cech widzimy coś ciekawego (w kontekscie hipotezy) - taneczność jest w sumie ważniejsza od speechiness - po usunięciu speechiness i tak sobie będzie bardzo dobrze radził. Za to energiczość i nastrój (valence) bardzo znikomo się różnią - w przeciwieństwie do tego co założyłem w hipotezie - myślałem że, po usunięciu speechiness, będą jednymi z najważniejszych wskaźników.

W takim razie zróbmy pare modeli: 
1. "Próbka testowa" - Ze wszystkimi cechami
2. Model do sprawdzenia hipotezy (bez speechiness i explicit)
3. Model też bez danceability
4. Model bez samego danceability (tak w sumie z ciekawości)
5. Samo speechiness i explicit
6. Samo speechiness

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Zbior treningowy - 80%,a testowy 20% (bez walidacyjnego narazie bo tu wziąłem domyślny LogisticRegression(), nie będę zmieniał sam wag)
X = df[wszystkie_cechy]
y = df["czy_hiphop"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

warianty = {
    "1. Wszystkie cechy": wszystkie_cechy,
    "2. Bez speechiness i explicit": [c for c in wszystkie_cechy if c not in ["speechiness", "explicit"]],
    "3. Bez speechiness, explicit i danceability": [c for c in wszystkie_cechy if c not in ["speechiness", "explicit", "danceability"]],
    "4. Bez danceability": [c for c in wszystkie_cechy if c not in ["danceability"]],
    "5. Samo speechiness i explicit": [c for c in wszystkie_cechy if c in ["speechiness", "explicit"]],
    "6. Samo speechiness": [c for c in wszystkie_cechy if c in ["speechiness"]],
}

wyniki = []
wagi_wszystkich = {}
for nazwa, kolumny in warianty.items():
    # Standaryzacja Z-score tylko (!) na zbiorze treningowym (zeby nie było data leakage)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train[kolumny])
    X_te = scaler.transform(X_test[kolumny])

    model = LogisticRegression(random_state=42)
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    wyniki.append({
        "Wariant": nazwa,
        "Accuracy": round(accuracy_score(y_test, y_pred), 3),
        "Precision": round(precision_score(y_test, y_pred), 3),
        "Recall": round(recall_score(y_test, y_pred), 3),
        "F1-score": round(f1_score(y_test, y_pred), 3),
    })
    wagi_wszystkich[nazwa] = pd.Series(model.coef_[0], index=kolumny)

tabela_wag = pd.DataFrame(wagi_wszystkich).sort_values(by="1. Wszystkie cechy", ascending=False)
print("<<< Wagi cech we wszystkich modelach (|cecha| wiekszy -> waga wazniejsza, NaN - cecha usunieta) >>>")
print(tabela_wag.round(3))

print("<<< Porownanie skutecznosci modeli >>>")
print(pd.DataFrame(wyniki).to_string(index=False))

<<< Wagi cech we wszystkich modelach (|cecha| wiekszy -> waga wazniejsza, NaN - cecha usunieta) >>>
                  1. Wszystkie cechy  2. Bez speechiness i explicit  3. Bez speechiness, explicit i danceability  4. Bez danceability  5. Samo speechiness i explicit  6. Samo speechiness
speechiness                    1.595                            NaN                                          NaN                1.996                           1.843                1.979
danceability                   1.552                          1.768                                          NaN                  NaN                             NaN                  NaN
loudness                       0.474                          0.338                                        0.468                0.588                             NaN                  NaN
acousticness                   0.446                          0.349                                        0.121                0.256                   

## Wnioski ##
1. Wagi: 
    - Największą wagę dostało speechiness (1.595) i zaraz po tym danceability (1.552)
    - Explicit ma o dziwo ponad 4x mniejszą wagę niż speechiness - nawet mniejszą od akustyczności i żywości
    - Pomimo, że korelacja valence była bliska 0, to model, w połączeniu z danceability używa nastroju - ale odwrotnie (nastrój & ~tanecznosc => ~hip-hop). Potwierdza to fakt, że kiedy zabieramy taneczność, waga nastroju spada blisko zera
2. Weryfikacja hipotezy: 
    - Po zabraniu speechiness i explicit, Accuracy spadło zaledwie z 0.806 do 0.775, a F1-score z 0.818 do 0.797
    - Potwierdza to drugą cz. hipotezy - bez informacji o ratio słów i wylgaryźmie model nadal potrafi dobrze określić gatunek
    - Co ciekawe, model oparty o samo speechiness - tylko jedną cechę, ma lepsze wyniki od tego opartego na speechiness i explicit (Acc .747 vs .723 i F1-score .750 vs .729), a nawet działa lepiej niż model oparty o wszystkie cechy oprócz danceability (aż 10 cech). Dodanie explicit bez danceability wręcz przeszkadza modelowi, bo większość hiphipu w tym zbiorze nie ma znaczka explicit. Natomiast przy samym speechiness jako najważniejszej cesze, całą reszta cech o słabych wagach, działa bardziej jako szum, dlatego model 4 działa gorzej od 5 i 6.
3. Co ratuje model bez speechiness? 
    - Hipoteza zakładała, że po zabraniu speechiness model oprze się na danceability, energy i valence. 
    - 3. model pokazuje jednak, że tylko danceability podtrzymuje wysokie wyniki - też bez tej cechy model ślepnieje, Accuracy oraz F1-score leci na łeb na szyję (Acc do .573 - a skoro w zbiorze jest ok. połowa hiphopu to praktycznie rzut monetą)
    - Zabranie danceability w 4. obniża Acc znacznie bardziej niż brak speechiness i explicit - do 0.708, a Recall spada z .775 do .690. Dzieje sie tak przez te indyjskie piosenki, które są melodyjne (ich śpiewanie) - są w hiphopie a mają niskie speechiness, przez co bez danceability model wrzuca je do rocka


Zobaczmy tylko jeszcze jak nasze testy by wyglądały gdybyśmy spróbowali oczyścić nasze dane z hinduskich mismatchy.
Usuniemy wykonawców z bollywood i popu hindi (llm ich wyszukał) - zazwyczaj oni będą mieli złe przypisania.

In [20]:
# lista do usuniecia
bollywood = [
    "Anirudh Ravichander", "Sachin-Jigar", "Mustafa Zahid", "Pritam",
    "A.R. Rahman", "Vishal-Shekhar", "Shreya Ghoshal", "Atif Aslam",
    "Bilal Saeed", "Hiphop Tamizha", "Guru Randhawa", "Badshah", "Hardy Sandhu"
]
wzorzec = "|".join(bollywood)

df_clean = df[~((df["track_genre"] == "hip-hop") & (df["artists"].str.contains(wzorzec, case=False, na=False)))].copy()

print(f"Usunieto {len(df) - len(df_clean)}, zostalo:")
print(df_clean["track_genre"].value_counts())
print("Nowe srednie speechiness:")
print(df_clean.groupby("track_genre")["speechiness"].agg(["mean", "median"]).round(3))

X_c = df_clean[wszystkie_cechy]
y_c = df_clean["czy_hiphop"]
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)

wyniki_clean = []
for nazwa, kolumny in warianty.items():
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_c[kolumny])
    X_te = scaler.transform(X_te_c[kolumny])

    model = LogisticRegression(random_state=42)
    model.fit(X_tr, y_tr_c)
    y_pred = model.predict(X_te)

    wyniki_clean.append({
        "Wariant": nazwa,
        "Accuracy": round(accuracy_score(y_te_c, y_pred), 3),
        "Precision": round(precision_score(y_te_c, y_pred), 3),
        "Recall": round(recall_score(y_te_c, y_pred), 3),
        "F1-score": round(f1_score(y_te_c, y_pred), 3),
    })
print()
print()
print("<<< Porownanie skutecznosci po usunieciu >>>")
print(pd.DataFrame(wyniki_clean).to_string(index=False))

Usunieto 131, zostalo:
track_genre
hip-hop    579
rock       554
Name: count, dtype: int64
Nowe srednie speechiness:
              mean  median
track_genre               
hip-hop      0.147   0.108
rock         0.052   0.039


<<< Porownanie skutecznosci po usunieciu >>>
                                    Wariant  Accuracy  Precision  Recall  F1-score
                         1. Wszystkie cechy     0.872      0.892   0.853     0.872
              2. Bez speechiness i explicit     0.828      0.853   0.802     0.827
3. Bez speechiness, explicit i danceability     0.559      0.561   0.638     0.597
                        4. Bez danceability     0.775      0.835   0.698     0.761
             5. Samo speechiness i explicit     0.749      0.798   0.681     0.735
                        6. Samo speechiness     0.749      0.839   0.629     0.719


Teraz zbiór jest prawie idealnie zbalandowany: 579 hiphop na 554 rock.
- Acc głównego modelu (wszystkie cechy) skoczyła z .806 aż do .872,
- Statystyki modelu bez dancebility też skoczyły, acc z .708 do .775
- Po wyrzuceniu tych hinduskich ballad explicit przestał psuć wynik: Acc modeli 5. i 6. są takie same, a model 5. (speechiness i explicit) ma większe F1-score o .016